# Historical runtime reference — public adaptation

Read README.md in this directory first. Default is PREFLIGHT_ONLY. Requires authorized external competition/model assets and2×T4. Supplied historical output was audited; this public copy has not been GPU rerun. No data or weights are embedded.


# LifeMate W50 + ModelA + PhoWhisper-medium — V2, 16/11/2026

**Input / Add Input (PRIVATE):**
1. BTC: `nguyenminhtric/zalo-ai`, đã giải nén `train+public_test/`.
2. Base LLM: Kaggle Models `nguyenminhtric/qwen3-5-4b` (Add Input → Models).
3. Adapter W50: `nguyenminhtric/modela`, chứa `63.8.zip` hoặc adapter r64 đã giải nén.
4. ASR: dataset PRIVATE `nguyenminhtric/update-medium` chứa asset gọn PhoWhisper-medium epoch5. Notebook tìm cả ZIP và best_adapter đã giải nén.

Chọn **GPU T4 ×2**, bật Internet cho cài thư viện, kiểm SHA base Kaggle qua metadata publisher và tải base PhoWhisper đã khóa revision. Không cần input S5b62.75/Zipformer. Không train lại, không tải/chạy notebook ngoài. Inference worker dùng weights local, tắt tải mạng.

LLM là **Qwen3.5-4B + adapter W50 r64/alpha128 đúng SHA của63.8.zip**, không phải nhánh Qwen38. ASR đúng **epoch5 / foldA**, train695 nguồn; không gọi là full-fit1391. Beam5/max200 token như benchmark trong medium.zip (WER current13,15% là kết quả lưu trước, **không phải số đo notebook này**).

ASR current+history chạy trước trên2T4 → cache có audio/model/decode hash → tiến trình ASR thoát → W50 chia2shard. Một original ToolEnv/mẫu, đủ nhật ký call/result, không replay history, không đọc/copy state files. V5B81,21 chỉ là mốc người dùng báo, không là input hoặc nhãn của notebook.

`45s` mặc định là budget nghiên cứu trênT4. Báo thời gian **tổng hai stage**, không giả là đo online A30 đạt30s. ProfileA30 dùng1A30/budget28s; vẫn cần kiểm bản online/Docker cuối. Không có score mới cho tới khi chạy và chấm thực tế.


In [ ]:
# ===== CONFIG: sửa tại đây; input được kiểm trước mọi cài đặt/tải model =====
from pathlib import Path
RUN_ID = 'w50_modela_medium_v2'  # đổi khi đổi model/code/decode/budget; không ghi đè planner đang dở
HARDWARE_PROFILE = 't4x2_development'  # hoặc 'a30_final': đúng1A30
STAGE = 'PREFLIGHT_ONLY'  # ALL / ASR_ONLY / LLM_ONLY / FINALIZE_ONLY / PREFLIGHT_ONLY
SPLIT = 'public_test'  # chọn rõ ràng, không tự nhảy sang private_test
LIMIT = None  # smoke tùy chọn; có LIMIT thì output là PARTIAL, không phải full634
DATA_ROOT_OVERRIDE = None
LLM_ADAPTER_INPUT = None  # mặc định chỉ tìm nguyenminhtric/modela
ASR_ADAPTER_INPUT = None  # tự tìm đúng SHA trong nguyenminhtric/update-medium
ASR_CACHE_INPUT = None  # ZIP LifeMate_ASR_CACHE_PRIVATE.zip từ chính notebook này nếu chạy LLM_ONLY
BASE_LLM_DIR = None  # tự tìm base dưới Kaggle Models nguyenminhtric/qwen3-5-4b; không phải W50 đã ghép
BASE_ASR_DIR = None
ALLOW_DOWNLOAD = True  # Internet chỉ dùng khi setup/prepare; worker inference offline
ASR_BEAMS = 5
ASR_MAX_NEW_TOKENS = 200
SAMPLE_BUDGET_S = 45.0 if HARDWARE_PROFILE == 't4x2_development' else 28.0
OUT = Path('/kaggle/working') / RUN_ID
TEMP = Path('/kaggle/temp/lifemate_w50_medium') / RUN_ID
CONFIG = dict(output_dir=str(OUT),temp_dir=str(TEMP),hardware_profile=HARDWARE_PROFILE,
 n_shards=2 if HARDWARE_PROFILE=='t4x2_development' else 1,split=SPLIT,limit=LIMIT,
 data_root_override=DATA_ROOT_OVERRIDE,llm_adapter_input=LLM_ADAPTER_INPUT,asr_adapter_input=ASR_ADAPTER_INPUT,
 asr_cache_input=ASR_CACHE_INPUT,base_llm_dir=BASE_LLM_DIR,base_asr_dir=BASE_ASR_DIR,
 allow_download=ALLOW_DOWNLOAD,cleanup_downloaded_bases=True,stage=STAGE,asr_beams=ASR_BEAMS,asr_max_new_tokens=ASR_MAX_NEW_TOKENS,
 sample_budget_s=SAMPLE_BUDGET_S,
 agent=dict(max_steps=12,max_new_tokens=512,context_limit=16384,policy='min',layout='v2',guard='off',stop_after_ask=True),
 engine=dict(gpu_memory_utilization=0.90,enforce_eager=True,prefix_caching=True,max_num_batched_tokens=2048,max_num_seqs=1))
if STAGE not in ('ALL','ASR_ONLY','LLM_ONLY','FINALIZE_ONLY','PREFLIGHT_ONLY'): raise ValueError('Invalid STAGE')
if LIMIT is not None and (type(LIMIT) is not int or LIMIT<=0): raise ValueError('LIMIT must be a positive integer or None')
print('BUILD: LIFEMATE_W50_MODELA_PHOWHISPER_MEDIUM_2T4_V2_20261116', STAGE, HARDWARE_PROFILE)


In [ ]:
# ===== Source tự chứa, kiểm SHA trước khi ghi file =====
import base64,hashlib,io,json,sys,zipfile
from pathlib import PurePosixPath
_B64 = '...'
_EXPECTED = '4933c1eed983f0a4e22eaed413020011bdcc60ec24138df1bf5ba0dd71b55234'
_raw=base64.b64decode(_B64)
assert hashlib.sha256(_raw).hexdigest()==_EXPECTED
CODE=TEMP/'code';CODE.mkdir(parents=True,exist_ok=True)
with zipfile.ZipFile(io.BytesIO(_raw)) as _z:
 for _n in _z.namelist():
  _p=PurePosixPath(_n)
  if _p.is_absolute() or '..' in _p.parts or _p.suffix!='.py': raise ValueError('Unsafe embedded source')
  _target=CODE.joinpath(*_p.parts);_target.parent.mkdir(parents=True,exist_ok=True);_target.write_bytes(_z.read(_n))
sys.path.insert(0,str(CODE))
print('Source SHA:',_EXPECTED,'modules:',len(list(CODE.rglob('*.py'))))


In [ ]:
# ===== 1. Kiểm input/hash/2T4, trước pip hoặc downloads =====
from pipeline import configure,stage_asr,stage_llm,finalize,bundle
CFG=configure(CONFIG)
print('Input PASS. Không có lời giải hoặc tool calls nào được tạo ở bước này.')


## 2. ASR trước — current và toàn bộ user history

Hai worker, mỗi GPU một bản PhoWhisper-medium. Merge adapter trênCPU theo đúng công thức LoRA; kiểm288 projections và số tham số. Không cần cài Unsloth để inference. ASR dùng môi trường HF4.57.6 riêng, giữ Torch/CUDA của Kaggle; không xung đột vLLM.

ASR_ONLY dừng sau khi có `LifeMate_ASR_CACHE_PRIVATE.zip`. Có thể tải ZIP này lên PRIVATE rồi dùng `ASR_CACHE_INPUT`/LLM_ONLY ở lượt khác; notebook kiểm đúng audio/indices/model/beam/token/source/version, không nhận cache Zipformer.


In [ ]:
if STAGE in ('ALL','ASR_ONLY'):
 try:
  CFG=stage_asr(CFG)
 except BaseException:
  bundle(CFG,failed=True)
  raise


## 3. W50 — ghép adapter63.8 với pinned Qwen3.5-4B rồi chạy native tools

Không ghép adapterW50 lên model S5b đã ghép; không trộn W50 thêm một lần nữa. Merge transaction chỉ được dùng khi `MERGE_READY.json` đủ file/hash. Giữ policy min/layoutv2/12bước/512token/greedy, seed20261031. Eager và scheduler nhỏ giảm rủi ro startup OOM trênT4, không bảo đảm score giống Docker cũ.

Không tự chạy lại mẫu đã có một ToolEnv dở. Khi worker lỗi, tải diagnostic ZIP; các shard hoàn tất có marker/hash có thể được xác minh và bỏ qua mà không replay. Không sửa submission theo từng public ID hoặc lấy V5B làm đáp án.


In [ ]:
if STAGE in ('ALL','LLM_ONLY'):
 try:
  CFG=stage_llm(CFG)
 except BaseException:
  bundle(CFG,failed=True)
  raise


In [ ]:
# ===== 4. Gộp đúng thứ tự, kiểm hash/log/call và xuất ZIP =====
if STAGE in ('ALL','LLM_ONLY','FINALIZE_ONLY'):
 SUMMARY=finalize(CFG)


## 5. Tải output

`LifeMate_W50_MEDIUM_RESULTS_PRIVATE.zip`: submission, timing, ASR cache, call/result ledger, agent events, model/input manifests và logs; không có base weights/audio/environment. Giữ ZIP riêng tư. `submission.jsonl` có đúng3field: id/tool_calls/state_hash. Full public phải đủ634 khi đúng phiên bản raw và LIMIT=None; không gọi LIMIT smoke là full submission.

Nếu cần Docker/offline: đây mới là notebook inference/chuẩn bị asset; chưa xuất image và chưa chứng minh deadline onlineA30. WER chỉ đo được trên TRAIN dev có reference/split phù hợp, không tính WER public bằng nhãn tự đoán.


In [ ]:
from IPython.display import display,FileLink
import os,shutil
for _name in ('submission.jsonl','LifeMate_W50_MEDIUM_RESULTS_PRIVATE.zip','LifeMate_ASR_CACHE_PRIVATE.zip','LifeMate_W50_MEDIUM_DIAGNOSTICS_PRIVATE.zip'):
 _p=OUT/_name
 if _p.exists():
  _easy=Path('/kaggle/working')/_name
  if _p.resolve()!=_easy.resolve(): shutil.copy2(_p,_easy)
  print(_easy,'bytes:',_easy.stat().st_size)
  display(FileLink(os.path.relpath(_easy,Path.cwd())))
print('Save Version → Output cũng có các ZIP ở thư mục /kaggle/working.')
